# Phase 4 — Baseline với 377 đặc trưng raw

Notebook này đọc và phân tích kết quả baseline `MultinomialNB` đã chạy trên ba split Phase 2.

- Mọi thí nghiệm dùng toàn bộ **377** triệu chứng nhị phân.
- Chỉ đánh giá trên **validation set**; test set được giữ lại cho Phase 7.
- Notebook không huấn luyện lại mô hình và không thay đổi dữ liệu hay artifact.

## Thiết lập

Chạy notebook từ thư mục `notebooks/` hoặc từ thư mục gốc repository.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
RESULT_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "results"
    / "baseline_377_features_schema_corrected"
    / "model_comparison_results.json"
)

if not RESULT_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy baseline artifact: {RESULT_PATH}")

with RESULT_PATH.open(encoding="utf-8") as file:
    results = json.load(file)

print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact: {RESULT_PATH}")


Project root: C:\Users\Admin\.vscode\ProjectCV\Medical_chatbot
Artifact: C:\Users\Admin\.vscode\ProjectCV\Medical_chatbot\experiments\results\baseline_377_features_schema_corrected\model_comparison_results.json


## Kiểm tra tính toàn vẹn của phiên baseline

Các assert dưới đây xác nhận đây là phiên `MultinomialNB`, dùng validation để đánh giá và có đủ 377 feature theo thứ tự đã lưu.

In [2]:
metadata = results["metadata"]
assert metadata["algorithms"] == ["MultinomialNB"]
assert metadata["evaluation_set"] == "validation (test set reserved for Phase 7)"

for experiment_id, experiment in results["results"].items():
    assert experiment["feature_count"] == 377, experiment_id
    assert len(experiment["feature_names"]) == 377, experiment_id
    assert experiment["algorithms"]["MultinomialNB"]["status"] == "success", experiment_id

pd.Series({
    "Timestamp": metadata["timestamp"],
    "Random seed": metadata["random_state"],
    "Target": metadata["target_col"],
    "Evaluation set": metadata["evaluation_set"],
    "Algorithm": metadata["algorithms"][0],
})

Timestamp                         2026-11-01T22:01:19.684203
Random seed                                               42
Target                                              diseases
Evaluation set    validation (test set reserved for Phase 7)
Algorithm                                      MultinomialNB
dtype: object

## Bảng kết quả validation

`Train–Validation gap` được giữ để phát hiện dấu hiệu overfitting, nhưng lựa chọn giữa các split dựa trên metric validation.

In [3]:
rows = []
for experiment_id, experiment in results["results"].items():
    baseline = experiment["algorithms"]["MultinomialNB"]
    validation = baseline["val_metrics"]
    rows.append({
        "Experiment": experiment_id,
        "Train samples": experiment["train_shape"][0],
        "Validation samples": experiment["val_shape"][0],
        "Features": experiment["feature_count"],
        "Top-1 accuracy": validation["top_1_accuracy"],
        "Macro-F1": validation["macro_f1"],
        "Weighted-F1": validation["weighted_f1"],
        "Top-3 accuracy": validation["top_3_accuracy"],
        "Top-5 accuracy": validation["top_5_accuracy"],
        "Train–Validation gap": baseline["overfitting_gap"],
        "Fit time (s)": baseline["fit_time_seconds"],
        "Representation": baseline["input_representation"],
    })

summary = pd.DataFrame(rows).sort_values("Macro-F1", ascending=False).reset_index(drop=True)
metric_columns = [
    "Top-1 accuracy", "Macro-F1", "Weighted-F1",
    "Top-3 accuracy", "Top-5 accuracy", "Train–Validation gap",
]
summary.style.format({column: "{:.2%}" for column in metric_columns}).format({"Fit time (s)": "{:.2f}"})

AttributeError: The '.style' accessor requires jinja2

## So sánh trực quan

Macro-F1 là metric ưu tiên để phản ánh hiệu năng cân bằng hơn giữa các lớp có tần suất khác nhau.

In [ ]:
plot_data = summary.melt(
    id_vars="Experiment",
    value_vars=["Top-1 accuracy", "Macro-F1", "Weighted-F1", "Top-3 accuracy", "Top-5 accuracy"],
    var_name="Metric",
    value_name="Score",
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_data, x="Metric", y="Score", hue="Experiment", ax=ax)
ax.set_title("Phase 4 baseline: validation metrics with 377 raw features")
ax.set_xlabel(None)
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter("{x:.0%}")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## Ghi nhận để quyết định bước tiếp theo

- `exp1_full` có Macro-F1 validation cao nhất trong baseline này; `exp3_capped` đứng sau.
- Cả ba phiên đều có train–validation gap thấp với baseline MultinomialNB.
- 49 feature zero-variance vẫn được giữ nguyên theo quyết định hiện tại. Chưa thực hiện feature selection, giảm chiều, tuning hay lựa chọn mô hình cuối cùng.
- Không dùng các kết quả này để khẳng định khả năng chẩn đoán y khoa hoặc đánh giá lâm sàng.

Bước tiếp theo cần được người dùng phê duyệt: chọn split tham chiếu để chạy so sánh thuật toán Phase 5, hoặc yêu cầu một thí nghiệm đặc trưng mới.